In [1]:
import ROOT
import numpy as np

# 1. Setup the File and Tree
f = ROOT.TFile.Open("tumor.root")
tree = f.Get("t")

# 2. Use Vector Buffers (Fixed for the "Double_t vs vector<double>" error)
energy_vec = ROOT.std.vector('double')()
vlm_vec = ROOT.std.vector('int')()

# 3. Link the buffers
tree.SetBranchAddress("et", energy_vec)
tree.SetBranchAddress("vlm", vlm_vec)

# 4. Define ONLY Detectors 4, 5, and 6
det_pos = {
    4: (10.0, 0.0, 0.0),    # Detector on +X
    6: (-10.0, 0.0, 0.0),   # Detector on -X
    5: (0.0, -10.0, 0.0)    # Detector on -Y
}

# 3D Histogram for the image
h3 = ROOT.TH3F("h3", "Reconstruction (4,5,6);X;Y;Z", 50, -10, 10, 50, -10, 10, 50, -10, 10)

# 5. The Stable Loop
num_entries = tree.GetEntries()
print(f"Processing {num_entries} events for detectors 4, 5, and 6...")

for i in range(num_entries):
    tree.GetEntry(i)
    
    # We must loop through the vector even if it only has 1 hit
    for j in range(energy_vec.size()):
        energy = energy_vec[j]
        vlm_id = vlm_vec[j]
        
        # Energy cut (Photopeak)
        if 0.60 < energy < 0.70:
            if vlm_id in det_pos:
                pos = det_pos[vlm_id]
                # Back-projection: Fill along the line to origin
                for t in np.linspace(0, 1, 100):
                    h3.Fill((1-t)*pos[0], (1-t)*pos[1], (1-t)*pos[2])

# 6. Draw
c = ROOT.TCanvas("c", "Reconstruction", 800, 600)
h3.SetFillColor(ROOT.kRed)
h3.Draw("ISO") 
c.SaveAs("result.png")

Processing 50000 events for detectors 4, 5, and 6...


Info in <TCanvas::Print>: png file result.png has been created
